# 00 - Exploration et nettoyage du dataset

**Dataset** : Jigsaw Toxic Comment Classification Challenge (Kaggle)

**Objectif** : Explorer le dataset, comprendre sa structure, identifier les problemes de qualite et nettoyer les donnees avant la modelisation.

## Sommaire
1. Chargement des donnees
2. Inspection generale
3. Analyse des labels
4. Analyse du texte
5. Nettoyage
6. Sauvegarde du dataset nettoye

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
RAW_PATH = '../data/raw/'
PROCESSED_PATH = '../data/processed/'

## 1. Chargement des donnees

In [ ]:
train = pd.read_csv(RAW_PATH + 'train.csv')
print(f"Shape: {train.shape}")
train.head()

## 2. Inspection generale

In [ ]:
print("=== Info ===")
print(f"Nombre de lignes : {len(train)}")
print(f"Nombre de colonnes : {len(train.columns)}")
print(f"\nColonnes : {list(train.columns)}")
print(f"\nTypes :\n{train.dtypes}")

In [ ]:
print("=== Valeurs manquantes ===")
print(train.isnull().sum())
print(f"\nTotal de valeurs manquantes : {train.isnull().sum().sum()}")

In [ ]:
print("=== Doublons ===")
print(f"Doublons (id) : {train.duplicated(subset='id').sum()}")
print(f"Doublons (comment_text) : {train.duplicated(subset='comment_text').sum()}")

## 3. Analyse des labels

In [ ]:
# Distribution des labels
label_counts = train[LABEL_COLS].sum().sort_values(ascending=False)
label_pct = (train[LABEL_COLS].mean() * 100).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(label_counts.index, label_counts.values, color='steelblue')
axes[0].set_title('Nombre de commentaires par label')
axes[0].set_xlabel('Nombre')

axes[1].barh(label_pct.index, label_pct.values, color='coral')
axes[1].set_title('Pourcentage de commentaires par label')
axes[1].set_xlabel('%')

plt.tight_layout()
plt.savefig('../artifacts/figures/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nDistribution des labels :")
for label in LABEL_COLS:
    count = train[label].sum()
    pct = train[label].mean() * 100
    print(f"  {label:20s} : {count:6d} ({pct:.2f}%)")

In [ ]:
# Commentaires sans aucun label ("clean")
train['n_labels'] = train[LABEL_COLS].sum(axis=1)
clean_count = (train['n_labels'] == 0).sum()
toxic_count = (train['n_labels'] > 0).sum()

print(f"Commentaires 'clean' (aucun label) : {clean_count} ({clean_count/len(train)*100:.1f}%)")
print(f"Commentaires toxiques (>=1 label)  : {toxic_count} ({toxic_count/len(train)*100:.1f}%)")

# Distribution du nombre de labels par commentaire
fig, ax = plt.subplots(figsize=(8, 5))
train['n_labels'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Nombre de labels par commentaire')
ax.set_xlabel('Nombre de labels')
ax.set_ylabel('Nombre de commentaires')
plt.tight_layout()
plt.show()

In [ ]:
# Matrice de co-occurrence des labels
co_occurrence = train[LABEL_COLS].T.dot(train[LABEL_COLS])

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(co_occurrence, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('Matrice de co-occurrence des labels')
plt.tight_layout()
plt.savefig('../artifacts/figures/label_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Analyse du texte

In [ ]:
# Longueur des commentaires (en caracteres et en mots)
train['char_length'] = train['comment_text'].str.len()
train['word_count'] = train['comment_text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

train['char_length'].hist(bins=100, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribution de la longueur des commentaires (caracteres)')
axes[0].set_xlabel('Nombre de caracteres')
axes[0].set_ylabel('Frequence')
axes[0].axvline(train['char_length'].median(), color='red', linestyle='--', label=f"Mediane: {train['char_length'].median():.0f}")
axes[0].legend()

train['word_count'].hist(bins=100, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Distribution de la longueur des commentaires (mots)')
axes[1].set_xlabel('Nombre de mots')
axes[1].set_ylabel('Frequence')
axes[1].axvline(train['word_count'].median(), color='red', linestyle='--', label=f"Mediane: {train['word_count'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.savefig('../artifacts/figures/text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Statistiques sur la longueur (mots) :")
print(train['word_count'].describe())

In [ ]:
# Longueur des commentaires : toxiques vs clean
fig, ax = plt.subplots(figsize=(10, 5))

train[train['n_labels'] == 0]['word_count'].hist(
    bins=80, ax=ax, alpha=0.6, color='green', label='Clean', density=True
)
train[train['n_labels'] > 0]['word_count'].hist(
    bins=80, ax=ax, alpha=0.6, color='red', label='Toxique', density=True
)

ax.set_title('Distribution de la longueur : toxiques vs clean')
ax.set_xlabel('Nombre de mots')
ax.set_ylabel('Densite')
ax.legend()
ax.set_xlim(0, 500)
plt.tight_layout()
plt.show()

In [ ]:
# Wordclouds par categorie
try:
    from wordcloud import WordCloud

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for i, label in enumerate(LABEL_COLS):
        text = ' '.join(train[train[label] == 1]['comment_text'].astype(str).values)
        wc = WordCloud(width=600, height=400, max_words=100, background_color='white',
                       colormap='Reds').generate(text)
        axes[i].imshow(wc, interpolation='bilinear')
        axes[i].set_title(label, fontsize=14)
        axes[i].axis('off')

    plt.suptitle('Wordclouds par categorie de toxicite', fontsize=16)
    plt.tight_layout()
    plt.savefig('../artifacts/figures/wordclouds_by_label.png', dpi=150, bbox_inches='tight')
    plt.show()
except ImportError:
    print("wordcloud non installe, skip wordclouds.")

## 5. Nettoyage

In [ ]:
def clean_text(text):
    """Nettoie un commentaire."""
    if not isinstance(text, str):
        return ''
    # Suppression des balises HTML
    text = re.sub(r'<[^>]+>', ' ', text)
    # Suppression des URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Suppression des adresses IP
    text = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', ' ', text)
    # Remplacement des retours a la ligne par des espaces
    text = re.sub(r'\n+', ' ', text)
    # Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text)
    # Strip
    text = text.strip()
    return text

print("Nettoyage en cours...")
df = train[['id', 'comment_text'] + LABEL_COLS].copy()
df['comment_text'] = df['comment_text'].apply(clean_text)
print("Nettoyage termine.")

In [ ]:
# Suppression des doublons sur le texte
n_before = len(df)
df = df.drop_duplicates(subset='comment_text', keep='first')
n_after = len(df)
print(f"Doublons supprimes : {n_before - n_after}")
print(f"Taille apres deduplication : {n_after}")

In [ ]:
# Suppression des commentaires vides apres nettoyage
n_before = len(df)
df = df[df['comment_text'].str.len() > 0].reset_index(drop=True)
n_after = len(df)
print(f"Commentaires vides supprimes : {n_before - n_after}")
print(f"Taille finale : {n_after}")

In [ ]:
# Verification apres nettoyage
print("=== Verification finale ===")
print(f"Shape : {df.shape}")
print(f"Valeurs manquantes : {df.isnull().sum().sum()}")
print(f"Doublons (texte) : {df.duplicated(subset='comment_text').sum()}")
print(f"\nDistribution des labels apres nettoyage :")
for label in LABEL_COLS:
    count = df[label].sum()
    pct = df[label].mean() * 100
    print(f"  {label:20s} : {count:6d} ({pct:.2f}%)")

## 6. Sauvegarde du dataset nettoye

In [ ]:
import os
os.makedirs(PROCESSED_PATH, exist_ok=True)

df.to_csv(PROCESSED_PATH + 'cleaned.csv', index=False)
print(f"Dataset nettoye sauvegarde dans {PROCESSED_PATH}cleaned.csv")
print(f"Shape finale : {df.shape}")

In [ ]:
# Resume
print("=" * 60)
print("RESUME DE L'EXPLORATION ET DU NETTOYAGE")
print("=" * 60)
print(f"Dataset original     : {len(train)} lignes")
print(f"Dataset nettoye      : {len(df)} lignes")
print(f"Lignes supprimees    : {len(train) - len(df)}")
print(f"Labels               : {LABEL_COLS}")
print(f"\nProchaine etape : split en train/val/test (src/preprocessing.py)")